# G4 — Adding an encoder of unrelated lineage to the frozen hub

Section C.13.2 states the gap plainly: DINOv2-S, -B and -L are distilled
from a common ViT-g/14 teacher, so zero-shot head transfer between them
demonstrates consistency **across model scales within one family**, not
across independent vision lineages. This notebook closes that gap.

**The test.** Encode a vision encoder with no DINOv2 ancestry over the
SAME image list, fit one linear map into the **frozen** hub — basis and
all existing entry maps untouched — and reuse the **same head**,
unchanged. Then compare its retention against the 93–96% measured
within the family.

**Which encoder.** The default is SigLIP 2, because it is already used
elsewhere in this project and has an entirely separate lineage (different
teacher, data, objective and organisation). `MobileCLIP-S1` and
`CLIP-ViT` are drop-in alternatives. A convolutional encoder (ConvNeXt)
is the strongest option and also a drop-in — it removes attention and
patch tokenisation entirely, so a head that transfers there shows the
shared space is not an artefact of one architecture family.

**On pooling.** DINOv2 is read out as CLS concatenated with the patch
mean. That exact recipe does not exist for every architecture, so each
encoder is read out the way its own designers intended — SigLIP by its
trained pooling head, ConvNeXt by global average pooling. Forcing a
common recipe across architectures would be less faithful, not more.
The hub's job is precisely to absorb such differences.

**Cost.** One encoding pass. No training, no fine-tuning, and the hub is
never rebuilt.

In [ ]:
import os
from pathlib import Path
STORAGE   = "drive"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"
try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - using LOCAL_DIR")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())
DATA_DIR = Path(os.environ["DATA_DIR"]); DATA_DIR.mkdir(exist_ok=True)
import torch
DEV = ("cuda" if torch.cuda.is_available()
       else "mps" if torch.backends.mps.is_available() else "cpu")
print("DATA_DIR:", DATA_DIR, "| device:", DEV)

In [ ]:
import numpy as np, json, zipfile, io, urllib.request
from concurrent.futures import ThreadPoolExecutor

# ---- which encoder to add. All are lineage-independent of DINOv2. ----
NEW_ENCODER = "google/siglip2-base-patch16-224"
# alternatives, drop-in:
#   "apple/MobileCLIP-S1"            (ViT, contrastive, separate lineage)
#   "openai/clip-vit-base-patch16"   (ViT, a third lineage)
#   "facebook/convnext-base-224"     (CONVOLUTIONAL - the strongest test)

_tag = NEW_ENCODER.split("/")[-1]
CKPT = DATA_DIR / f"e1_img_ckpt_{_tag}_native.npz"   # G1/G3 pick this up

# ---- the SAME id list every other cache used ----
zf = DATA_DIR / "annotations_trainval2017.zip"
assert zf.exists(), "annotations zip missing - run E1 or B1 once first"
with zipfile.ZipFile(str(zf)) as z:
    ann = json.load(z.open("annotations/captions_train2017.json"))
url = {im["id"]: im["coco_url"] for im in ann["images"]}
caps = {}
for a in ann["annotations"]:
    caps.setdefault(a["image_id"], []).append(a["caption"].strip())

# match the row count of the narrowest existing cache, so every space
# in the hub still describes the same items
_ref = sorted(DATA_DIR.glob("e1_img_ckpt_dinov2-*_cls+patch.npz"))
assert _ref, "no DINOv2 caches found - run E1 first"
N_PAIRS = min(len(np.load(str(f))["img"]) for f in _ref)
ids = sorted(set(caps) & set(url))[:N_PAIRS]
print(f"{len(ids)} ids, matched to the narrowest existing cache "
      f"({N_PAIRS} rows)")
print("this is the SAME deterministic prefix E1 used - that is what")
print("makes the new space row-aligned with the others")

In [ ]:
from transformers import AutoModel, AutoImageProcessor
from PIL import Image

def _fetch(i):
    for _ in range(3):
        try:
            raw = urllib.request.urlopen(url[ids[i]], timeout=20).read()
            return i, Image.open(io.BytesIO(raw)).convert("RGB")
        except Exception:
            pass
    return i, None

def encode_all(batch=64, save_every=1024):
    if CKPT.exists():
        d = np.load(str(CKPT))
        done, IMG = int(d["done"][0]), d["img"]
        print(f"resuming from row {done} ({IMG.shape})")
    else:
        done, IMG = 0, None
    if done >= len(ids):
        print("already complete"); return IMG[:len(ids)]

    proc = AutoImageProcessor.from_pretrained(NEW_ENCODER)
    mod = AutoModel.from_pretrained(NEW_ENCODER).to(DEV).eval()
    tower = getattr(mod, "vision_model", mod)     # SigLIP/CLIP -> tower

    with ThreadPoolExecutor(16) as pool:
        for start in range(done, len(ids), batch):
            end = min(start + batch, len(ids))
            got = dict(pool.map(_fetch, range(start, end)))
            imgs = [got[i] for i in range(start, end)]
            ok = [im for im in imgs if im is not None]
            if len(ok) < len(imgs):
                ok += [ok[-1]] * (len(imgs) - len(ok))   # rare fetch miss
            with torch.no_grad():
                b = proc(images=ok, return_tensors="pt").to(DEV)
                out = tower(**b)
                # each encoder read out the way its designers intended
                if getattr(out, "pooler_output", None) is not None:
                    v = out.pooler_output
                else:
                    v = out.last_hidden_state.mean(1)     # ConvNeXt etc.
            v = v.float().cpu().numpy()
            IMG = v if IMG is None else np.vstack([IMG, v])
            if (end % save_every == 0) or end == len(ids):
                np.savez_compressed(str(CKPT), img=IMG.astype(np.float32),
                                    done=np.array([end]),
                                    keep=np.array(ids[:end]))
                print(f"  {end}/{len(ids)} rows ({IMG.shape[1]}-d)")
    del mod
    if DEV == "cuda": torch.cuda.empty_cache()
    return IMG

IMG_NEW = encode_all()
print(f"\n{_tag}: {IMG_NEW.shape}  saved to {CKPT.name}")
print("G1 and G3 will pick this file up automatically on their next run.")

## The test: one map into the FROZEN hub, the SAME head unchanged

Nothing below refits the hub basis, the existing entry maps, or the
head. The only new object is one linear map from the new encoder into
the existing hub — which is exactly what the claim requires.

In [ ]:
# rebuild the hub EXACTLY as G1 did (same seed, same split, same width),
# then freeze it and add only the new encoder's entry map
SPACES = {}
for size in ("small", "base", "large"):
    f = DATA_DIR / f"e1_img_ckpt_dinov2-{size}_cls+patch.npz"
    if f.exists():
        SPACES[f"img_{size}"] = np.load(str(f))["img"].astype(np.float64)[:N_PAIRS]
d = np.load(str(DATA_DIR / "crossmodal_pairs.npz"))
SPACES["txt_bge"] = d["txt"].astype(np.float64)[:N_PAIRS]

HUB_DIM, ALPHA, TRAIN_ON = 512, 1e-2, "img_small"
rng = np.random.default_rng(0)
perm = rng.permutation(N_PAIRS); te, tr = perm[:1000], perm[1000:]
def l2n(V): return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
def ridge(X, Y, a=ALPHA):
    return np.linalg.solve(X.T @ X + a * np.eye(X.shape[1]), X.T @ Y)
def recall(S):
    o = np.argsort(-S, 1); r = (o == np.arange(len(S))[:, None]).argmax(1)
    return {k: float((r < k).mean()) for k in (1, 5, 10)}

_ref = np.hstack([(SPACES[k][tr] - SPACES[k][tr].mean(0)) /
                  (SPACES[k][tr].std(0).mean() + 1e-12) for k in SPACES])
_mu = _ref.mean(0)
_u, _sv, _VT = np.linalg.svd(_ref - _mu, full_matrices=False)
BASIS = _VT[:HUB_DIM].T / (_sv[:HUB_DIM] / np.sqrt(len(_ref)))
HUB_TR = (_ref - _mu) @ BASIS
TO_HUB = {k: ridge(SPACES[k][tr], HUB_TR) for k in SPACES}
HEAD = ridge(SPACES[TRAIN_ON][tr] @ TO_HUB[TRAIN_ON], SPACES["txt_bge"][tr])
GAL = l2n(SPACES["txt_bge"][te])
print(f"hub rebuilt and now FROZEN: {HUB_DIM}-d, {len(SPACES)} spaces, "
      f"head trained on {TRAIN_ON}")

# ---- the only new fit in this notebook ----
NEW = IMG_NEW.astype(np.float64)[:N_PAIRS]
TO_HUB_NEW = ridge(NEW[tr], HUB_TR)          # one linear map, closed form

print(f"\n{'encoder':22s} {'lineage':>22s} {'R@1':>7s} {'% of native':>12s}")
for enc in [k for k in SPACES if k.startswith("img_")] + [_tag]:
    if enc == _tag:
        P = l2n((NEW[te] @ TO_HUB_NEW) @ HEAD)
        nat = ridge(NEW[tr], SPACES["txt_bge"][tr])
        rn = recall(l2n(NEW[te] @ nat) @ GAL.T)
        lin = "INDEPENDENT"
    else:
        P = l2n((SPACES[enc][te] @ TO_HUB[enc]) @ HEAD)
        nat = ridge(SPACES[enc][tr], SPACES["txt_bge"][tr])
        rn = recall(l2n(SPACES[enc][te] @ nat) @ GAL.T)
        lin = "ViT-g student" + (" (trained here)" if enc == TRAIN_ON else "")
    r = recall(P @ GAL.T)
    print(f"{enc:22s} {lin:>22s} {r[1]:7.3f} "
          f"{100*r[1]/max(rn[1],1e-9):11.1f}%")

R = np.random.default_rng(9).standard_normal(TO_HUB_NEW.shape) / \
    np.sqrt(NEW.shape[1])
print(f"\ncontrol - random map for the new encoder: "
      f"R@1 {recall(l2n((NEW[te] @ R) @ HEAD) @ GAL.T)[1]:.3f} "
      f"(chance {1/len(te):.3f})")

## How to read it

Compare the new encoder's **% of native** against the 95.9 and 92.9 per
cent measured for DINOv2-base and -large in Section C.13.

- **Near that range** → the claim generalises from model scale to
  lineage. With a convolutional encoder, it generalises to architecture
  as well, which is the strongest version available.
- **Clearly below** → the drop measures what shared ancestry was
  contributing. That is a result, not a failure, and it should be
  reported with the same weight as a success: it would mean the hub's
  portability is partly a family property.
- **At chance** → check the control first, then the shape-agreement
  diagnostic in G3. A pair with no shape agreement has no correspondence
  for any hub to carry.

Either outcome belongs in Section C.13.2, which currently records this
test as untested.